In [3]:
import os
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import lightgbm as lgb
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

TARGETS = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
PROPERTY_WEIGHTS = {'Tg': 1.0, 'FFV': 10.0, 'Tc': 1.0, 'Density': 1.0, 'Rg': 1.0}
MLM_MODEL = 'DeepChem/ChemBERTa-77M-MLM'
MTR_MODEL = 'DeepChem/ChemBERTa-77M-MTR'
DATA_DIR  = '/kaggle/input/competitions/neurips-open-polymer-prediction-2025'
N_FOLDS   = 5

# Data Preprocessing

In [5]:
def canonicalize(smi):
    try:
        return Chem.MolToSmiles(Chem.MolFromSmiles(smi))
    except:
        return None

train = pd.read_csv(f'/kaggle/input/competitions/neurips-open-polymer-prediction-2025/train.csv')
test  = pd.read_csv(f'/kaggle/input/competitions/neurips-open-polymer-prediction-2025/test.csv')
ds1   = pd.read_csv(f'/kaggle/input/competitions/neurips-open-polymer-prediction-2025/train_supplement/dataset1.csv').rename(columns={'TC_mean': 'Tc'})
ds3   = pd.read_csv(f'/kaggle/input/competitions/neurips-open-polymer-prediction-2025/train_supplement/dataset3.csv')
ds4   = pd.read_csv(f'/kaggle/input/competitions/neurips-open-polymer-prediction-2025/train_supplement/dataset4.csv')

for df in [train, test, ds1, ds3, ds4]:
    df['canon'] = df['SMILES'].apply(canonicalize)

train_canons = set(train['canon'].dropna())

ds1_new = ds1[~ds1['canon'].isin(train_canons)][['canon', 'Tc']].dropna().reset_index(drop=True)
ds3_new = ds3[~ds3['canon'].isin(train_canons)][['canon', 'Tg']].dropna().reset_index(drop=True)
ds4_new = ds4[~ds4['canon'].isin(train_canons)][['canon', 'FFV']].dropna().reset_index(drop=True)

print(f'New supplement rows: Tc={len(ds1_new)}, Tg={len(ds3_new)}, FFV={len(ds4_new)}')

supp_frames = []
for df, col in [(ds1_new, 'Tc'), (ds3_new, 'Tg'), (ds4_new, 'FFV')]:
    tmp = pd.DataFrame({'canon': df['canon'].values})
    for t in TARGETS:
        tmp[t] = df[col].values if t == col else np.nan
    supp_frames.append(tmp)

all_train = pd.concat(
    [train[['canon'] + TARGETS]] + supp_frames,
    ignore_index=True
)

print(f'Combined train rows: {len(all_train)}')
print('Label counts:', all_train[TARGETS].notna().sum().to_dict())

New supplement rows: Tc=130, Tg=46, FFV=862
Combined train rows: 9011
Label counts: {'Tg': 557, 'FFV': 7892, 'Tc': 867, 'Density': 613, 'Rg': 614}


In [6]:
def load_or_compute(cache_path, smiles_list, model_name, batch_size=32):
    if os.path.exists(cache_path):
        print(f'Loading from cache: {cache_path}')
        return np.load(cache_path)
    print(f'Extracting with {model_name} ...')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).eval()
    all_embs = []
    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i : i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=512, return_tensors='pt')
        with torch.no_grad():
            out = model(**enc)
        emb  = out.last_hidden_state                          # (B, L, H)
        mask = enc['attention_mask'].unsqueeze(-1).float()    # (B, L, 1)
        pooled = (emb * mask).sum(1) / mask.sum(1).clamp(min=1e-9)  # (B, H)
        all_embs.append(pooled.numpy())
        if (i // batch_size) % 20 == 0:
            print(f'  {i + len(batch)}/{len(smiles_list)} done', end='\r')
    print()
    del model
    embs = np.vstack(all_embs)
    np.save(cache_path, embs)
    print(f'Saved to {cache_path}, shape: {embs.shape}')
    return embs

train_smiles = all_train['canon'].fillna('').tolist()
test_smiles  = test['canon'].fillna('').tolist()

mlm_train = load_or_compute('mlm_train_embs.npy', train_smiles, MLM_MODEL)
mlm_test  = load_or_compute('mlm_test_embs.npy',  test_smiles,  MLM_MODEL)
mtr_train = load_or_compute('mtr_train_embs.npy', train_smiles, MTR_MODEL)
mtr_test  = load_or_compute('mtr_test_embs.npy',  test_smiles,  MTR_MODEL)

print(f'Train embs: MLM={mlm_train.shape}, MTR={mtr_train.shape}')
print(f'Test embs:  MLM={mlm_test.shape},  MTR={mtr_test.shape}')

Extracting with DeepChem/ChemBERTa-77M-MLM ...


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/14.0M [00:00<?, ?B/s]

  8992/9011 done
Saved to mtr_train_embs.npy, shape: (9011, 384)
Extracting with DeepChem/ChemBERTa-77M-MTR ...


Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MTR
Key                             | Status     | 
--------------------------------+------------+-
norm_mean                       | UNEXPECTED | 
regression.dense.weight         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
regression.out_proj.weight      | UNEXPECTED | 
regression.dense.bias           | UNEXPECTED | 
norm_std                        | UNEXPECTED | 
regression.out_proj.bias        | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  3/3 done
Saved to mtr_test_embs.npy, shape: (3, 384)
Train embs: MLM=(9011, 384), MTR=(9011, 384)
Test embs:  MLM=(3, 384),  MTR=(3, 384)


In [1]:
def compute_wmae(results):
    total_w = sum(PROPERTY_WEIGHTS.values())
    return sum(results[t] * PROPERTY_WEIGHTS[t] for t in TARGETS) / total_w


def run_cv(X_embs, df, label, regressor_type='ridge'):
    """
    5-fold OOF CV per target on the labeled subset.
    regressor_type: 'ridge' or 'lgbm'
    """
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    results = {}

    for target in TARGETS:
        mask = df[target].notna().values
        X = X_embs[mask]
        y = df[target].values[mask]

        oof = np.zeros(len(y))
        for tr_idx, val_idx in kf.split(X):
            if regressor_type == 'ridge':
                scaler = StandardScaler()
                X_tr  = scaler.fit_transform(X[tr_idx])
                X_val = scaler.transform(X[val_idx])
                model = Ridge(alpha=1.0)
                model.fit(X_tr, y[tr_idx])
                oof[val_idx] = model.predict(X_val)
            else:  # lgbm
                model = lgb.LGBMRegressor(
                    n_estimators=1000, learning_rate=0.05, num_leaves=63,
                    subsample=0.8, colsample_bytree=0.8, random_state=42,
                    n_jobs=1, verbose=-1
                )
                model.fit(
                    X[tr_idx], y[tr_idx],
                    eval_set=[(X[val_idx], y[val_idx])],
                    callbacks=[
                        lgb.early_stopping(50, verbose=False),
                        lgb.log_evaluation(-1)
                    ]
                )
                oof[val_idx] = model.predict(X[val_idx])

        mae = np.abs(y - oof).mean()
        results[target] = mae
        print(f'  [{label}] {target}: n={len(y)}, OOF MAE={mae:.4f}')

    results['wMAE'] = compute_wmae(results)
    print(f'  [{label}] OOF wMAE = {results["wMAE"]:.4f}')
    return results

# Experiment 1: ChemBERTa-77M-MLM + Ridge Regression

In [ ]:
print('=' * 55)
print('Experiment 1: ChemBERTa-77M-MLM + Ridge Regression')
print('=' * 55)
res_mlm_ridge = run_cv(mlm_train, all_train, label='MLM+Ridge', regressor_type='ridge')

# Experiment 2: ChemBERTa-77M-MLM + LightGBM

In [ ]:
print('=' * 55)
print('Experiment 2: ChemBERTa-77M-MLM + LightGBM')
print('=' * 55)
res_mlm_lgbm = run_cv(mlm_train, all_train, label='MLM+LGBM', regressor_type='lgbm')

# Experiment 3: ChemBERTa-77M-MTR + Ridge Regression

In [ ]:
print('=' * 55)
print('Experiment 3: ChemBERTa-77M-MTR + Ridge Regression')
print('=' * 55)
res_mtr_ridge = run_cv(mtr_train, all_train, label='MTR+Ridge', regressor_type='ridge')

# Experiment 4: ChemBERTa-77M-MTR + LightGBM

In [ ]:
print('=' * 55)
print('Experiment 4: ChemBERTa-77M-MTR + LightGBM')
print('=' * 55)
res_mtr_lgbm = run_cv(mtr_train, all_train, label='MTR+LGBM', regressor_type='lgbm')

# Evaluation

In [ ]:
cols = TARGETS + ['wMAE']

chemberta_rows = {
    'MLM + Ridge (baseline)': res_mlm_ridge,
    'MLM + LightGBM':         res_mlm_lgbm,
    'MTR + Ridge':            res_mtr_ridge,
    'MTR + LightGBM':         res_mtr_lgbm,
}

team_baselines = {
    'Morgan FP+TF-IDF (Combined)': {
        'Tg': 51.36, 'FFV': 0.0060, 'Tc': 0.0285,
        'Density': 0.0432, 'Rg': 1.5793, 'wMAE': 3.791
    },
    'RDKit Descriptors': {
        'Tg': 51.28, 'FFV': 0.0072, 'Tc': 0.0275,
        'Density': 0.0332, 'Rg': 2.00, 'wMAE': None
    },
}

df_results   = pd.DataFrame(chemberta_rows).T[cols]
df_baselines = pd.DataFrame(team_baselines).T[cols]
df_all = pd.concat([df_results, df_baselines])
df_all.index.name = 'Method'

pd.set_option('display.float_format', '{:.4f}'.format)
display(df_all)